In [2]:


# STEP 1: Import Libraries

import pandas as pd
import numpy as np

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# STEP 2: Load Cleaned Dataset
# ============================================================

file_path = "/content/Cleaned_Sample_Dataset.xlsx"

df = pd.read_excel(file_path)

print("Dataset Loaded Successfully")
print("Shape:", df.shape)


# ============================================================
# STEP 3: Basic Dataset Check
# ============================================================

print("\nFirst 5 Rows:")
print(df.head())

print("\nData Types:")
print(df.dtypes)


# ============================================================
# STEP 4: Prepare Date Column
# ============================================================

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date")

df = df.reset_index(drop=True)

print("\nDate Column Prepared Successfully")


# ============================================================
# STEP 5: Create Monthly Time Series
# ============================================================

monthly_data = (
    df.groupby("Date", as_index=False)["Sales"]
      .sum()
      .sort_values("Date")
)

monthly_data = monthly_data.set_index("Date")

series = monthly_data["Sales"].dropna()

print("\nMonthly Sales Data:")
print(monthly_data.head())


# ============================================================
# STEP 6: Descriptive Statistics
# ============================================================

print("\nDescriptive Statistics:")

print(series.describe())


# ============================================================
# STEP 7: Calculate 3-Month Moving Average
# ============================================================

monthly_data["Moving_Average_3M"] = (
    series.rolling(window=3).mean()
)

print("\n3-Month Moving Average:")
print(monthly_data.tail(10))


# ============================================================
# STEP 8: Time Series Decomposition
# ============================================================

decomposition = seasonal_decompose(
    series,
    model="additive",
    period=12
)

trend = decomposition.trend

seasonal = decomposition.seasonal

residual = decomposition.resid

print("\nTime Series Decomposition Completed")


# ============================================================
# STEP 9: Analyze Seasonality
# ============================================================

seasonal_summary = (
    seasonal
    .groupby(seasonal.index.month)
    .mean()
)

seasonal_summary = seasonal_summary.reset_index()

seasonal_summary.columns = [
    "Month",
    "Average_Seasonal_Effect"
]

print("\nSeasonal Pattern:")
print(seasonal_summary)


# ============================================================
# STEP 10: Stationarity Test
# ============================================================

adf_result = adfuller(series)

adf_statistic = adf_result[0]

p_value = adf_result[1]

print("\nADF Test Results")

print("ADF Statistic:", adf_statistic)

print("P-Value:", p_value)

if p_value < 0.05:

    stationarity_result = "Stationary"

else:

    stationarity_result = "Non-Stationary"

print("Result:", stationarity_result)


# ============================================================
# STEP 11: Train-Test Split
# ============================================================

train_size = int(len(series) * 0.80)

train = series.iloc[:train_size]

test = series.iloc[train_size:]

print("\nTrain Records:", len(train))

print("Test Records:", len(test))


# ============================================================
# STEP 12: Create Naive Baseline
# ============================================================

naive_forecast = pd.Series(
    train.iloc[-1],
    index=test.index
)


# ============================================================
# STEP 13: Naive Model Evaluation
# ============================================================

naive_mae = mean_absolute_error(
    test,
    naive_forecast
)

naive_rmse = np.sqrt(
    mean_squared_error(
        test,
        naive_forecast
    )
)


def calculate_mape(actual, predicted):

    actual = np.array(actual)

    predicted = np.array(predicted)

    return np.mean(
        np.abs(
            (actual - predicted) / actual
        )
    ) * 100


naive_mape = calculate_mape(
    test,
    naive_forecast
)


print("\nNaive Baseline Results")

print("MAE:", naive_mae)

print("RMSE:", naive_rmse)

print("MAPE:", naive_mape)


# ============================================================
# STEP 14: Build SARIMA Model
# ============================================================

model = SARIMAX(

    train,

    order=(1, 1, 1),

    seasonal_order=(1, 1, 1, 12),

    enforce_stationarity=False,

    enforce_invertibility=False

)


# ============================================================
# STEP 15: Fit SARIMA Model
# ============================================================

model_fit = model.fit(
    disp=False
)

print("\nSARIMA Model Fitted Successfully")


# ============================================================
# STEP 16: Forecast Test Data
# ============================================================

test_forecast_result = model_fit.get_forecast(
    steps=len(test)
)

test_forecast = (
    test_forecast_result
    .predicted_mean
)


# ============================================================
# STEP 17: SARIMA Model Evaluation
# ============================================================

sarima_mae = mean_absolute_error(
    test,
    test_forecast
)

sarima_rmse = np.sqrt(
    mean_squared_error(
        test,
        test_forecast
    )
)

sarima_mape = calculate_mape(
    test,
    test_forecast
)


print("\nSARIMA Model Results")

print("MAE:", sarima_mae)

print("RMSE:", sarima_rmse)

print("MAPE:", sarima_mape)


# ============================================================
# STEP 18: Compare Models
# ============================================================

model_comparison = pd.DataFrame({

    "Model": [
        "Naive Baseline",
        "SARIMA"
    ],

    "MAE": [
        naive_mae,
        sarima_mae
    ],

    "RMSE": [
        naive_rmse,
        sarima_rmse
    ],

    "MAPE": [
        naive_mape,
        sarima_mape
    ]

})


print("\nModel Comparison:")

print(model_comparison)


# ============================================================
# STEP 19: Select Best Model
# ============================================================

best_model = model_comparison.loc[
    model_comparison["RMSE"].idxmin(),
    "Model"
]


print("\nBest Model:", best_model)


# ============================================================
# STEP 20: Train Final SARIMA Model
# ============================================================

final_model = SARIMAX(

    series,

    order=(1, 1, 1),

    seasonal_order=(1, 1, 1, 12),

    enforce_stationarity=False,

    enforce_invertibility=False

)


# ============================================================
# STEP 21: Fit Final Model
# ============================================================

final_model_fit = final_model.fit(
    disp=False
)

print("\nFinal Model Fitted Successfully")


# ============================================================
# STEP 22: Generate Future Forecast
# ============================================================

forecast_periods = 12

future_result = final_model_fit.get_forecast(
    steps=forecast_periods
)

future_forecast = (
    future_result.predicted_mean
)

future_conf_int = (
    future_result.conf_int()
)


# ============================================================
# STEP 23: Create Future Forecast Dates
# ============================================================

forecast_dates = pd.date_range(

    start=series.index.max()
    + pd.offsets.MonthBegin(1),

    periods=forecast_periods,

    freq="MS"

)


# ============================================================
# STEP 24: Create Forecast Table
# ============================================================

forecast_table = pd.DataFrame({

    "Date": forecast_dates,

    "Actual_Sales": np.nan,

    "Forecast_Sales":
        future_forecast.values,

    "Lower_Bound":
        future_conf_int.iloc[:, 0].values,

    "Upper_Bound":
        future_conf_int.iloc[:, 1].values,

    "Model": best_model

})


print("\nFuture Forecast:")

print(forecast_table)


# ============================================================
# STEP 25: Create Historical Table
# ============================================================

historical_table = pd.DataFrame({

    "Date": series.index,

    "Actual_Sales": series.values,

    "Forecast_Sales": np.nan,

    "Lower_Bound": np.nan,

    "Upper_Bound": np.nan,

    "Model": best_model

})


# ============================================================
# STEP 26: Combine Historical and Forecast Data
# ============================================================

final_forecast_output = pd.concat(

    [
        historical_table,
        forecast_table
    ],

    ignore_index=True

)


# ============================================================
# STEP 27: Create Analysis Summary
# ============================================================

summary = pd.DataFrame({

    "Metric": [

        "Total Historical Sales",

        "Average Monthly Sales",

        "Minimum Monthly Sales",

        "Maximum Monthly Sales",

        "Train Records",

        "Test Records",

        "ADF Statistic",

        "ADF P-Value",

        "Naive MAE",

        "Naive RMSE",

        "Naive MAPE",

        "SARIMA MAE",

        "SARIMA RMSE",

        "SARIMA MAPE",

        "Best Model",

        "Forecast Periods"

    ],

    "Value": [

        series.sum(),

        series.mean(),

        series.min(),

        series.max(),

        len(train),

        len(test),

        adf_statistic,

        p_value,

        naive_mae,

        naive_rmse,

        naive_mape,

        sarima_mae,

        sarima_rmse,

        sarima_mape,

        best_model,

        forecast_periods

    ]

})


# ============================================================
# STEP 28: Create Seasonal Analysis Table
# ============================================================

seasonal_analysis = seasonal_summary.copy()


# ============================================================
# STEP 29: Save ONLY ONE Output Excel File
# ============================================================

output_file = (
    "/content/Time_Series_Analysis_Output.xlsx"
)


with pd.ExcelWriter(
    output_file,
    engine="openpyxl"
) as writer:

    final_forecast_output.to_excel(

        writer,

        sheet_name="Forecast_Output",

        index=False

    )

    model_comparison.to_excel(

        writer,

        sheet_name="Model_Comparison",

        index=False

    )

    summary.to_excel(

        writer,

        sheet_name="Summary",

        index=False

    )

    seasonal_analysis.to_excel(

        writer,

        sheet_name="Seasonality",

        index=False

    )


# ============================================================
# STEP 30: Final Validation
# ============================================================

print("\n============================================")

print("TIME-SERIES ANALYSIS COMPLETED")

print("============================================")

print("Final Dataset Rows:", len(df))

print("Historical Months:", len(series))

print("Best Model:", best_model)

print("Forecast Periods:", forecast_periods)

print("Output File:", output_file)

print("============================================")

Dataset Loaded Successfully
Shape: (84, 10)

First 5 Rows:
        Date     Sales  Orders  Customers Region Product_Category  Year  \
0 2019-01-01  10322.86   126.0         64  North      Electronics  2019   
1 2019-02-01  10906.51   141.0         73   West  Office Supplies  2019   
2 2019-03-01  12172.61   145.0         90  North        Furniture  2019   
3 2019-04-01  13079.13   164.0         96   West        Furniture  2019   
4 2019-05-01  11792.19   115.0         77   West        Furniture  2019   

   Month Month_Name  Quarter  
0      1    January        1  
1      2   February        1  
2      3      March        1  
3      4      April        2  
4      5        May        2  

Data Types:
Date                datetime64[ns]
Sales                      float64
Orders                     float64
Customers                    int64
Region                      object
Product_Category            object
Year                         int64
Month                        int64
Month_Name 

/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)



SARIMA Model Fitted Successfully

SARIMA Model Results
MAE: 880.8308524939397
RMSE: 1230.7777270163501
MAPE: 5.056184322835996

Model Comparison:
            Model          MAE         RMSE      MAPE
0  Naive Baseline  1186.401176  1497.241767  6.821281
1          SARIMA   880.830852  1230.777727  5.056184

Best Model: SARIMA


/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)



Final Model Fitted Successfully

Future Forecast:
         Date  Actual_Sales  Forecast_Sales   Lower_Bound   Upper_Bound  \
0  2026-01-01           NaN    18116.297341  16229.696067  20002.898614   
1  2026-02-01           NaN    19264.519374  17372.697299  21156.341448   
2  2026-03-01           NaN    18832.898437  16940.690089  20725.106785   
3  2026-04-01           NaN    20133.277439  18241.096750  22025.458128   
4  2026-05-01           NaN    18736.053789  16843.867459  20628.240118   
5  2026-06-01           NaN    19743.871628  17851.681802  21636.061454   
6  2026-07-01           NaN    18761.807355  16869.612718  20654.001991   
7  2026-08-01           NaN    17633.330703  15741.130222  19525.531184   
8  2026-09-01           NaN    17227.169469  15334.961964  19119.376975   
9  2026-10-01           NaN    17050.912696  15158.696654  18943.128738   
10 2026-11-01           NaN    18011.168383  16118.940799  19903.395966   
11 2026-12-01           NaN    18535.862118  1664